# 00 — Shared Data and Final Comparison

**EN3150 Assignment 03 — Resource-Constrained CNN for Edge Image Classification**  
**Owner:** Sahanya  
**Environment:** Google Colab + TensorFlow/Keras

## Purpose
Use the first half now to verify the common dataset. Use the final-comparison half only after the four model result JSON files exist.

In [ ]:
%pip install -q tensorflow-datasets scikit-learn

In [ ]:
import os, gc, time, json, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
print('TensorFlow:',tf.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
SEED=42
IMG_SIZE=64
BATCH_SIZE=64
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print(SEED,IMG_SIZE,BATCH_SIZE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/EN3150_A03')
TFDS_ROOT=PROJECT_ROOT/'tfds_data'; ARTIFACT_ROOT=PROJECT_ROOT/'artifacts'; RESULT_ROOT=PROJECT_ROOT/'shared_results'; PLOT_ROOT=PROJECT_ROOT/'plots'
for p in [TFDS_ROOT,ARTIFACT_ROOT,RESULT_ROOT,PLOT_ROOT]: p.mkdir(parents=True,exist_ok=True)
print(PROJECT_ROOT)

## 1. Shared data preparation

In [ ]:
SPLITS=['train[:70%]','train[70%:85%]','train[85%:]']
(raw_train,raw_val,raw_test),ds_info=tfds.load('tf_flowers',split=SPLITS,as_supervised=True,with_info=True,data_dir=str(TFDS_ROOT),shuffle_files=False)
CLASS_NAMES=ds_info.features['label'].names
NUM_CLASSES=len(CLASS_NAMES)
def count_examples(ds): return int(tf.data.experimental.cardinality(ds).numpy())
print('Classes:',CLASS_NAMES)
print('Train:',count_examples(raw_train),'Val:',count_examples(raw_val),'Test:',count_examples(raw_test))

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE
def preprocess(image,label):
    image=tf.image.resize(image,[IMG_SIZE,IMG_SIZE],antialias=True)
    return tf.cast(image,tf.float32),label
train_ds=(raw_train.shuffle(2048,seed=SEED,reshuffle_each_iteration=True).map(preprocess,num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))
val_ds=(raw_val.map(preprocess,num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))
test_ds=(raw_test.map(preprocess,num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE))

In [ ]:
plt.figure(figsize=(9,9))
for images,labels in train_ds.take(1):
    for i in range(min(9,len(images))):
        plt.subplot(3,3,i+1); plt.imshow(tf.cast(images[i],tf.uint8)); plt.title(CLASS_NAMES[int(labels[i])]); plt.axis('off')
plt.tight_layout(); plt.savefig(PLOT_ROOT/'dataset_examples_64x64.png',dpi=180); plt.show()

In [ ]:
dataset_summary={'dataset':'tf_flowers','image_size':[64,64,3],'split':{'train':'70%','validation':'15%','test':'15%'},'seed':SEED,'classes':CLASS_NAMES,'train_samples':count_examples(raw_train),'validation_samples':count_examples(raw_val),'test_samples':count_examples(raw_test)}
with open(RESULT_ROOT/'dataset_summary.json','w') as f: json.dump(dataset_summary,f,indent=2)
print(json.dumps(dataset_summary,indent=2))

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
feat: add shared TF Flowers data preparation and split verification
```
File:
```text
notebooks/00_shared_data_and_final_comparison.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.

# STOP HERE until training is complete
Later this notebook expects `model_a.json`, `model_b.json`, `mobilenetv2.json`, and `efficientnetb0.json` in `MyDrive/EN3150_A03/shared_results/`.

## 2. Final comparison

In [ ]:
required=['model_a.json','model_b.json','mobilenetv2.json','efficientnetb0.json']
missing=[x for x in required if not (RESULT_ROOT/x).exists()]
if missing: raise FileNotFoundError('Missing: '+', '.join(missing))
records=[json.load(open(RESULT_ROOT/x)) for x in required]
comparison=pd.DataFrame(records)
display(comparison)
comparison.to_csv(RESULT_ROOT/'final_model_comparison.csv',index=False)

In [ ]:
plt.figure(figsize=(8,5)); plt.scatter(comparison.model_size_mb,comparison.accuracy,s=80)
for _,r in comparison.iterrows(): plt.annotate(r.model,(r.model_size_mb,r.accuracy),xytext=(5,5),textcoords='offset points')
plt.xlabel('Saved model size (MB)'); plt.ylabel('Test accuracy'); plt.title('Accuracy vs Model Size'); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/'final_accuracy_vs_model_size.png',dpi=180); plt.show()

In [ ]:
plt.figure(figsize=(8,5)); plt.scatter(comparison.parameters,comparison.accuracy,s=80)
for _,r in comparison.iterrows(): plt.annotate(r.model,(r.parameters,r.accuracy),xytext=(5,5),textcoords='offset points')
plt.xscale('log'); plt.xlabel('Parameters (log scale)'); plt.ylabel('Test accuracy'); plt.title('Accuracy vs Parameter Count'); plt.grid(alpha=.25); plt.tight_layout(); plt.savefig(PLOT_ROOT/'final_accuracy_vs_parameters.png',dpi=180); plt.show()

## Report discussion
Use the measured table to discuss accuracy, memory footprint, computational cost, Model A vs Model B, and custom Model B vs MobileNetV2/EfficientNetB0. Do not decide the conclusion before results exist.

---
### ✅ Git commit checkpoint
Do **not** commit every cell. Commit after this meaningful milestone.

Suggested commit:
```text
analysis: add final accuracy memory and computational cost comparison
```
File:
```text
notebooks/00_shared_data_and_final_comparison.ipynb
```
Beginner workflow: **Colab → File → Download `.ipynb` → replace local notebook → GitHub Desktop → Commit → Push**.